# Task 2 — Vanilla GAN on MNIST with Logistic Loss

Same architecture as Task 1 but with logistic (softplus) loss instead of BCE.
We compare generated samples at epochs 5, 10, and 50 against Task 1.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

import global_config as gc
from utils import device_check, show_generated_images, build_model_name
from GAN import Generator, Discriminator, train_GAN

## General setup

In [ ]:
LOG_WANDB = True
SEED = 1

# Device setup
device = device_check()

# Reproducibility
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True
    PIN_MEMORY = True
    NUM_WORKERS = 8
    PERSISTENT_WORKERS = True
    PREFETCH_FACTOR = 4
else:
    PIN_MEMORY = False
    NUM_WORKERS = 0
    PERSISTENT_WORKERS = False
    PREFETCH_FACTOR = 2

## Load MNIST dataset

In [ ]:
BATCH_SIZE = 256

transform = transforms.Compose([
    transforms.ToTensor()
])

train_dataset = datasets.MNIST(
    root=gc.DATA_DIR,
    train=True,
    transform=transform,
    download=True
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=PERSISTENT_WORKERS,
    prefetch_factor=PREFETCH_FACTOR,
)

print("Number of training samples:", len(train_dataset))
print("Number of batches per epoch:", len(train_loader))

## Visualize a few real images

In [ ]:
real_images, real_labels = next(iter(train_loader))

fig, axes = plt.subplots(1, 6, figsize=(10, 2))
for i in range(6):
    axes[i].imshow(real_images[i].squeeze(), cmap="gray")
    axes[i].set_title(f"Label: {real_labels[i].item()}")
    axes[i].axis("off")

plt.tight_layout()
plt.show()

## Initialize models

In [ ]:
LATENT_DIM = 100
G_HIDDEN_DIM = 128
D_HIDDEN_DIM = 128
IMAGE_DIM = 28 * 28

G = Generator(
    z_dim=LATENT_DIM,
    h_dim=G_HIDDEN_DIM,
    x_dim=IMAGE_DIM,
).to(device)

D = Discriminator(
    x_dim=IMAGE_DIM,
    h_dim=D_HIDDEN_DIM,
).to(device)

print(G)
print()
print(D)

## Define optimizers

No criterion object is needed — logistic loss is computed inside `train_GAN` via `F.softplus`.

In [ ]:
LR = 1e-3
BETAS = (0.9, 0.999)

g_optimizer = optim.Adam(G.parameters(), lr=LR, betas=BETAS)
d_optimizer = optim.Adam(D.parameters(), lr=LR, betas=BETAS)

## Train the GAN with logistic loss

In [ ]:
EPOCHS = 100

config = {
    "epochs": EPOCHS,
    "latent_dim": LATENT_DIM,
    "image_dim": IMAGE_DIM,
    "batch_size": BATCH_SIZE,
    "g_lr": LR,
    "d_lr": LR,
    "g_hidden_dim": G_HIDDEN_DIM,
    "d_hidden_dim": D_HIDDEN_DIM,
    "betas": BETAS,
    "optimizer": "Adam",
    "loss_type": "logistic",
    "dataset": "MNIST",
    "model": "Vanilla GAN - Logistic Loss",
    "seed": SEED,
}

wandb_kwargs = dict(
    entity=gc.WANDB_ENTITY,
    project=gc.WANDB_PROJECT,
    name="Vanilla GAN MNIST - Logistic Loss",
    tags=["Task 2", "MNIST", "Vanilla GAN", "Logistic Loss"],
    dir=str(gc.WANDB_DIR),
    config=config,
    mode="online" if LOG_WANDB else "disabled",
)

train_GAN(
    G=G,
    D=D,
    train_loader=train_loader,
    g_optimizer=g_optimizer,
    d_optimizer=d_optimizer,
    config=config,
    device=device,
    wandb_kwargs=wandb_kwargs,
    loss_type="logistic",
)

## Display final generated samples

In [ ]:
show_generated_images(G, LATENT_DIM, device)

## Save trained models

In [ ]:
checkpoint_path = gc.MODELS_DIR / build_model_name(config, task_name="task2")

torch.save({
    "generator_state_dict": G.state_dict(),
    "discriminator_state_dict": D.state_dict(),
    "g_optimizer_state_dict": g_optimizer.state_dict(),
    "d_optimizer_state_dict": d_optimizer.state_dict(),
    "config": config,
}, checkpoint_path)

print(f"Checkpoint saved to: {checkpoint_path}")